# 01 — Introduction to the Model Context Protocol (MCP)

This notebook introduces the **Model Context Protocol** — Anthropic's open standard for connecting AI models to external tools and data sources.

## Learning Objectives

- Understand what MCP is and why it exists
- Learn the three MCP primitives: Tools, Resources, Prompts
- See the difference between manual tool authoring and MCP
- Run a basic MCP server

## The Problem: Manual Tool Integration

Without MCP, integrating Claude with an external service requires:

1. **Writing JSON tool schemas** by hand
2. **Implementing tool functions** that call the external API
3. **Maintaining both** as the API evolves

For a single GitHub integration, you might need 20+ tool schemas and functions.

In [ ]:
# Example: Manual tool schema (what you'd write WITHOUT MCP)
manual_tool_schema = {
    "name": "create_github_issue",
    "description": "Create a new issue in a GitHub repository",
    "input_schema": {
        "type": "object",
        "properties": {
            "repo": {"type": "string", "description": "Repository name (owner/repo)"},
            "title": {"type": "string", "description": "Issue title"},
            "body": {"type": "string", "description": "Issue body"},
            "labels": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Labels to apply",
            },
        },
        "required": ["repo", "title"],
    },
}

print("Manual schema has", len(str(manual_tool_schema)), "characters")
print("Imagine writing 20 of these for one service...")

## The MCP Solution

MCP shifts tool definition from **your code** to a **dedicated MCP server**:

```
Without MCP:                          With MCP:

Your App                              Your App
├── schema: create_issue              └── MCP Client
├── func: create_issue()                   └── GitHub MCP Server
├── schema: list_repos                          ├── create_issue (pre-built)
├── func: list_repos()                          ├── list_repos (pre-built)
└── ... (20 more)                               └── ... (all pre-built)
```

Someone else (often the service provider) creates and maintains the MCP server.

## Three MCP Primitives

| Primitive | Controlled By | Purpose | Example |
|-----------|--------------|---------|--------|
| **Tools** | Model (Claude) | Actions Claude can invoke | `edit_document` |
| **Resources** | Application | Data for the app to read | `docs://documents` |
| **Prompts** | User | Workflow templates | `/format` slash command |

In [ ]:
# With MCP + FastMCP, the same tool becomes:
# (This is conceptual — we'll run real servers in later notebooks)

EXAMPLE_CODE = '''
from mcp.server.fastmcp import FastMCP
from pydantic import Field

mcp = FastMCP("GitHubMCP")

@mcp.tool(
    name="create_github_issue",
    description="Create a new issue in a GitHub repository",
)
def create_issue(
    repo: str = Field(description="Repository name (owner/repo)"),
    title: str = Field(description="Issue title"),
    body: str = Field(description="Issue body", default=""),
):
    # Call GitHub API here
    pass
'''

print("With FastMCP:")
print("- No manual JSON schema")
print("- Schema auto-generated from type hints")
print("- Descriptions from Field() annotations")
print(f"- Code: {len(EXAMPLE_CODE)} characters vs {len(str(manual_tool_schema))} for schema alone")

## MCP Architecture Overview

```
User
 │
 ▼
Application Server (your code)
 │
 ▼
MCP Client (SDK-provided interface)
 │  stdin/stdout, HTTP, or WebSocket
 ▼
MCP Server (tools + resources + prompts)
 │
 ▼
External Services (GitHub, Slack, DBs)
```

## Exercise

1. List 3 external services you use and describe what tools an MCP server for each might provide.
2. For each service, identify which capabilities would be Tools vs Resources vs Prompts.
3. Estimate how many manual tool schemas you'd need to write without MCP for those services.

In [ ]:
# Exercise: Fill in your answers

services = [
    {
        "name": "GitHub",
        "tools": ["create_issue", "merge_pr", "create_branch"],
        "resources": ["repos://list", "repos://{owner}/{repo}/issues"],
        "prompts": ["/review (code review workflow)"],
    },
    {
        "name": "Your Service 1",
        "tools": [],  # TODO: fill in
        "resources": [],  # TODO: fill in
        "prompts": [],  # TODO: fill in
    },
    {
        "name": "Your Service 2",
        "tools": [],  # TODO: fill in
        "resources": [],  # TODO: fill in
        "prompts": [],  # TODO: fill in
    },
]

for svc in services:
    total = len(svc["tools"]) + len(svc["resources"]) + len(svc["prompts"])
    print(f"{svc['name']}: {total} capabilities")
    print(f"  Tools: {svc['tools']}")
    print(f"  Resources: {svc['resources']}")
    print(f"  Prompts: {svc['prompts']}")
    print()